# Tucker / HOOI 基礎実装

このNotebookでは、HOSVDを初期値にして **HOOI (Higher-Order Orthogonal Iteration)** を自作する。

HOSVDは各modeのfactorを1回求めるのに対し、HOOIは他modeのfactorを使いながら各factorを反復更新する。

## このNotebookで自分で実装するもの

1. `mode 0` のfactorを1回更新する
2. `hooi_sweep()`：全modeを1回ずつ更新する
3. `core_from_factors()`：現在のfactorからcoreを計算する
4. `hooi()`：sweepを反復し、収束まで回す

それ以外の初期化・評価・可視化・TensorLyとの比較は用意してある。

## 完了条件

- HOOIの1 sweepを自分で書ける
- HOSVDとHOOIの違いを説明できる
- HOOIの誤差推移を確認できる
- 自作HOOIとTensorLyを再構成誤差で比較できる


## 1. 実験対象

3階テンソル `X` を、multilinear rank `(3, 2, 2)` で近似する。

ここではrank探索はしない。**同じrankでHOSVDとHOOIを比較すること**が目的。


In [ ]:
import matplotlib.pyplot as plt
import torch

from nn_compression.compression import (
    hosvd,
    reconstruct_tucker,
    truncated_svd,
)
from nn_compression.metrics import relative_frobenius_error
from nn_compression.tensor import mode_dot, unfold

torch.manual_seed(0)

X = torch.randn(6, 5, 4)
ranks = {0: 3, 1: 2, 2: 2}

print("shape:", tuple(X.shape))
print("ranks:", ranks)


## 2. HOSVDを初期値にする

まず既存の `hosvd(X, ranks)` を実行し、HOOIを始める前のfactor・core・再構成誤差を保存する。

### 確認すること

- factorのshapeが各modeの元dimensionと指定rankに対応している
- この誤差がHOOI比較の基準値になる


In [ ]:
core_hosvd, factors_hosvd = hosvd(X, ranks)
X_hat_hosvd = reconstruct_tucker(core_hosvd, factors_hosvd)
hosvd_error = relative_frobenius_error(X, X_hat_hosvd)

print("HOSVD core shape:", tuple(core_hosvd.shape))
for mode, U in factors_hosvd.items():
    print(f"mode={mode}: U.shape={tuple(U.shape)}")
print("HOSVD relative error:", float(hosvd_error))


## 3. 練習1：mode 0 のfactorを1回更新する

HOOIの最小単位を、まずmode 0だけで実装する。

### やること

**入力**
- 元テンソル `X`
- 現在のfactor `factors_work`
- 更新対象 `mode = 0`
- 目標rank `ranks[0]`

**出力**
- 更新後の `U0` を `updated_u0` に入れる

### 条件

- 更新対象自身のfactorは、そのmodeの更新には使わない
- `mode_dot`、`unfold`、`truncated_svd` を使う
- 最終的に `updated_u0.shape == (X.shape[0], ranks[0])` になることを確認する

具体的な処理順はここでは示さない。HOOIの定義から組み立てる。


In [ ]:
mode = 0

factors_work = {
    m: U.clone()
    for m, U in factors_hosvd.items()
}

# TODO: mode 0 のfactorを1回更新する
projected = None
updated_u0 = None

# 実装後に有効化
# assert updated_u0.shape == (X.shape[0], ranks[0])
# print("projected shape:", tuple(projected.shape))
# print("updated U0 shape:", tuple(updated_u0.shape))


## 4. 練習2：`hooi_sweep()` を実装する

次に、mode 0だけの更新を全modeへ一般化する。

### やること

`hooi_sweep(X, factors, ranks)` を完成させ、**各modeのfactorを1回ずつ更新**する。

**入力**
- `X`: 元テンソル
- `factors`: 現在のfactor辞書
- `ranks`: modeごとの目標rank

**出力**
- 全mode更新後のfactor辞書

### 条件

- 更新は `ranks` に含まれる各modeについて行う
- sweep途中で更新済みのfactorがあれば、その最新値を使う
- 元の `factors` を直接書き換えない


In [ ]:
def hooi_sweep(
    X: torch.Tensor,
    factors: dict[int, torch.Tensor],
    ranks: dict[int, int],
) -> dict[int, torch.Tensor]:
    updated_factors = {
        mode: U.clone()
        for mode, U in factors.items()
    }

    # TODO: 各modeのfactorを1回ずつ更新する
    pass


## 5. 練習3：`core_from_factors()` を実装する

factorを更新しただけではTucker分解は完成しない。現在のfactorに対応するcoreを元の `X` から計算する。

### やること

`core_from_factors(X, factors)` を完成させる。

**入力**
- `X`: 元テンソル
- `factors`: 現在のfactor辞書

**出力**
- Tucker core

### 確認すること

この設定では最終的なcore shapeが `(3, 2, 2)` になる。


In [ ]:
def core_from_factors(
    X: torch.Tensor,
    factors: dict[int, torch.Tensor],
) -> torch.Tensor:
    # TODO: 現在のfactorからcoreを計算する
    pass


## 6. 練習4：`hooi()` を実装する

最後に、HOSVD初期化からHOOIの反復までを1つの関数にまとめる。

### やること

`hooi(X, ranks, max_iter, tol)` を完成させる。

**処理の大枠**
- HOSVDで初期化する
- `hooi_sweep()` を反復する
- 各sweep後にcoreと再構成誤差を計算する
- 誤差の変化が十分小さくなったら停止する

**返り値**
- 最終core
- 最終factor辞書
- HOSVD初期値を含むrelative Frobenius errorの履歴 `history`

ここではループ内部の具体的なコードは示さない。


In [ ]:
def hooi(
    X: torch.Tensor,
    ranks: dict[int, int],
    max_iter: int = 20,
    tol: float = 1e-6,
) -> tuple[
    torch.Tensor,
    dict[int, torch.Tensor],
    list[float],
]:
    # TODO: HOSVD初期化から収束までHOOIを実行する
    pass


## 7. HOSVDと自作HOOIを比較する

ここからは評価用。上の3関数を実装したら実行する。

### 確認すること

- `history[0]` がHOSVDの誤差になっている
- sweep後の誤差がどのように変化するか
- 最終HOOI誤差がHOSVDと比べてどうなったか
- factorそのものではなく、**再構成結果と誤差**を比較する


In [ ]:
core_hooi, factors_hooi, history = hooi(
    X,
    ranks,
    max_iter=30,
    tol=1e-7,
)

X_hat_hooi = reconstruct_tucker(core_hooi, factors_hooi)
hooi_error = relative_frobenius_error(X, X_hat_hooi)

print("HOSVD error:", float(hosvd_error))
print("HOOI  error:", float(hooi_error))
print("iterations:", len(history) - 1)
print("history:", history)

assert len(history) >= 1
assert abs(history[0] - float(hosvd_error)) < 1e-6
assert tuple(core_hooi.shape) == tuple(ranks[mode] for mode in range(X.ndim))


## 8. 誤差の推移を可視化する

HOOIが反復法であることを、sweepごとのrelative Frobenius errorから確認する。


In [ ]:
plt.plot(range(len(history)), history, marker="o")
plt.xlabel("Sweep (0 = HOSVD initialization)")
plt.ylabel("Relative Frobenius error")
plt.title("HOSVD initialization -> HOOI")
plt.grid(True)
plt.show()


## 9. TensorLyと比較する

最後に、自作実装が大きくずれていないかをTensorLyで検証する。

TensorLyは**答えを作るためではなく検証用**として使う。

### 比較するもの

- core / factorのshape
- 再構成tensorのshape
- relative Frobenius error

factorの数値そのものは、符号や基底の取り方が異なるため完全一致を要求しない。


In [ ]:
import tensorly as tl
from tensorly.decomposition import tucker
from tensorly.tucker_tensor import tucker_to_tensor

tl.set_backend("pytorch")

rank_list = [ranks[mode] for mode in range(X.ndim)]

core_tl, factors_tl = tucker(
    X,
    rank=rank_list,
    init="svd",
    n_iter_max=100,
    tol=1e-8,
)

X_hat_tl = tucker_to_tensor((core_tl, factors_tl))
tensorly_error = relative_frobenius_error(X, X_hat_tl)

print("self HOSVD:", float(hosvd_error))
print("self HOOI :", float(hooi_error))
print("TensorLy  :", float(tensorly_error))

print("TensorLy core:", tuple(core_tl.shape))
for mode, U in enumerate(factors_tl):
    print(f"TensorLy mode={mode}: U.shape={tuple(U.shape)}")


## 10. 最後に整理する

実装と比較が終わったら、自分の言葉で次を説明する。

1. HOSVDとHOOIではfactorの求め方がどう違うか
2. HOOIで1つのfactorを更新するとき、なぜ他modeのfactorを使うのか
3. 1 sweepとは何か
4. なぜHOOIはHOSVDより計算量が増えるのか
5. 同じmultilinear rankでもHOOIを使う意味は何か

ここまで説明できれば、このNotebookは完了。
